In [1]:
#Modify the path to a directory on your machine
import os
os.environ["CRDS_PATH"] = "/home/hailin/Documents/CRDS"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Packages that allow us to get information about objects:
import asdf
import copy
import shutil

# Numpy library:
import numpy as np

# For downloading data
import requests

# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch

import matplotlib.pyplot as plt
import matplotlib as mpl

# Plotting tools:
from pipeline1_plotting_tools import download_files, plot_jump, plot_jumps, plot_ramp, plot_ramps, show_image, side_by_side

# Use this version for non-interactive plots (easier scrolling of the notebook)
%matplotlib inline

# Use this version (outside of Jupyter Lab) if you want interactive plots
#%matplotlib notebook

# List of possible data quality flags
from jwst.datamodels import dqflags

# The entire calwebb_detector1 pipeline
from jwst.pipeline import calwebb_detector1

# Individual steps that make up calwebb_detector1
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.superbias import SuperBiasStep
from jwst.ipc import IPCStep                                                                                    
from jwst.refpix import RefPixStep                                                                
from jwst.linearity import LinearityStep
from jwst.persistence import PersistenceStep
from jwst.dark_current import DarkCurrentStep
from jwst.jump import JumpStep
from jwst.ramp_fitting import RampFitStep
from jwst import datamodels

import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from pathlib import Path

import jwst
print(jwst.__version__)

1.19.2


In [3]:
import re
data_path = Path('../data/JWST/3000s_exposure')
data_behav_mask = np.zeros((30,7))
for item in data_path.iterdir():
    print('\\texttt{'+re.sub(r"_", r"\\_", item.stem)+'}\\\\')

\texttt{jw01121002001\_02102\_00001\_nrs2}\\
\texttt{jw01121120001\_02102\_00001\_nrs2}\\
\texttt{jw01121012001\_02102\_00001\_nrs2}\\
\texttt{jw01121102001\_03102\_00001\_nrs2}\\
\texttt{jw01121008001\_02102\_00001\_nrs2}\\
\texttt{jw01121116001\_02102\_00001\_nrs2}\\
\texttt{jw01121138001\_02102\_00001\_nrs2}\\
\texttt{jw01121006001\_02102\_00001\_nrs2}\\
\texttt{jw01121124001\_02102\_00001\_nrs2}\\
\texttt{jw01121004001\_02102\_00001\_nrs2}\\
\texttt{jw01121150001\_02102\_00001\_nrs2}\\
\texttt{jw01121106001\_02102\_00001\_nrs2}\\
\texttt{jw01121114001\_02102\_00001\_nrs2}\\
\texttt{jw01121118001\_02102\_00001\_nrs2}\\
\texttt{jw01121136001\_02102\_00001\_nrs2}\\
\texttt{jw01121154001\_02102\_00001\_nrs2}\\
\texttt{jw01121158001\_02102\_00001\_nrs2}\\
\texttt{jw01121110001\_02102\_00001\_nrs2}\\
\texttt{jw01121144001\_02102\_00001\_nrs2}\\
\texttt{jw01121134001\_02102\_00001\_nrs2}\\
\texttt{jw01121104001\_02102\_00001\_nrs2}\\
\texttt{jw01121130001\_02102\_00001\_nrs2}\\
\texttt{jw

In [2]:
crds_dir = '/home/hailin/Documents/CRDS/references/jwst/nirspec/'
data_path = Path('../data/JWST/3000s_exposure')
data_behav_mask = np.zeros((30,7))
i = 0
for item in data_path.iterdir():
    input_file_base = item.name
#    if os.path.exists('./results/blind2/'+ input_file_base +'.txt'):
#        continue
    jump_file = '../data/JWST/3000s_exposure/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    # gain file
    gain_file = crds_dir + jump.meta.ref_file.gain.name[7:]
    gain = fits.getdata(gain_file, 'SCI')
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    dn_range = range(dn_min, dn_max)
    dn_hist_bins = np.arange(dn_min-0.5, dn_max+0.5)
    total_pix = 2048 * 2048
    print('group number is {}'.format(n_group))

    # flag but no jump from pipeline
    flag_map = (jump.groupdq & ~dqflags.pixel['JUMP_DET'] > 0)
    flag_map_2d = np.sum(flag_map[0, :, :, :], axis=0)
    flag_pix = np.sum(flag_map_2d>0)

    raw_data = jump.data[0]
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]

    # dev：外加一个delta mask，相当于手搓版的jump mask
    delta_mask = np.zeros((2048, 2048), dtype=bool)
    delta_thr = 60
    delta = (raw_data - np.roll(raw_data, 1, axis=0))[1:,:,:]
    delta_map_2d = np.sum(np.abs(delta[:,:,:]) > delta_thr, axis=0)
    delta_mask   = (delta_map_2d > 0)
    delta_pix = np.sum(delta_map_2d > 0)

    # 结合 pipeline （没有jump）+ our delta
    dq_map_2d = flag_map_2d + delta_map_2d
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)

    low_gain_indexes = np.where(gain < 0.6)

    # c stands for control:
    c_bad_mosaic   = True
    c_secondary    = True
    c_edge         = True
    c_correlated   = True
    c_brightness   = True

    mosaic_size = 16
    mask_perc_thr_1  = 0.7
    secondary_size_1 = 3
    mask_perc_thr_2  = 0.4
    secondary_size_2 = 1
    edge_width = 64
    col_ratio = 0.5
    brightness_thr  = 100
    brightness_size = 2

    mask = np.zeros((2048, 2048), dtype=bool)

    # 这次我们直接从超亮pixel的分布入手，mask掉超亮pixel周围的pixel
    # 并且这次设计两套bad mask。越亮的区域需要的secondary越大。
    bad_mask = np.zeros((2048, 2048), dtype=bool)
    mosaic_area = mosaic_size**2
    mosaic_half = int(mosaic_size / 2)
    N_mosaic = int(2048 / mosaic_size)
    bad_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    bad_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_bad_mosaic:
        bad_mosaic_count = np.zeros((N_mosaic, N_mosaic))
        # 现在可以直接从dq_map_indexes出发，对其进行pixel内的计数
        bad_pix_map = (dq_map_2d > 0)
        # 热力图，但是不带已经flagged的
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                bad_mosaic_count[y,x] = np.sum(bad_pix_map[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size])
        # 把马赛克化的mask汇总于此：
        bad_mosaic_1  = (bad_mosaic_count/mosaic_area >mask_perc_thr_1)
        bad_mosaic_2  = (bad_mosaic_count/mosaic_area >mask_perc_thr_2)

    #cols_mosaic = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    #cols_mosaic[:,-14:-12] = True
    #cols_mosaic[:,28:30] = True
    #cols_mosaic[:,-4:-2] = True

    secondary_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool) #coule be the most important
    secondary_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_secondary:
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                if  bad_mosaic_1[y,x] :
                    y_min = np.max([y-secondary_size_1,0])
                    x_min = np.max([x-secondary_size_1,0])
                    y_max = np.min([y+secondary_size_1,N_mosaic-1])
                    x_max = np.min([x+secondary_size_1,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_1**2:
                                secondary_mosaic_1[y2,x2] = True
                if  bad_mosaic_2[y,x] :
                    y_min = np.max([y-secondary_size_2,0])
                    x_min = np.max([x-secondary_size_2,0])
                    y_max = np.min([y+secondary_size_2,N_mosaic-1])
                    x_max = np.min([x+secondary_size_2,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_2**2:
                                secondary_mosaic_2[y2,x2] = True

    mask_mosaic = bad_mosaic_1  + secondary_mosaic_1 + bad_mosaic_2  + secondary_mosaic_2

    for y in range(N_mosaic):  # jwst 指标先y后x
        for x in range(N_mosaic):
            if  mask_mosaic[y,x]:
                bad_mask[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size] = True

    edge_mask = np.zeros((2048, 2048), dtype=bool)
    if c_edge:
        edge_mask[0:edge_width,:] = True
        edge_mask[-edge_width:,:] = True
        edge_mask[:,0:edge_width] = True
        edge_mask[:,-edge_width:] = True

    # correlated noise for bad columns

    correlated_mask = np.zeros((2048, 2048), dtype=bool)
    if c_correlated:
        photo_zero_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
        noisy_indexes = np.where((photo_zero_mask > 20) & (photo_zero_mask <= 400))
        photo_zero_mask[dq_map_indexes] = 0
        photo_zero_mask[noisy_indexes] = 0
        n_col = N_mosaic - 2 * int(edge_width / mosaic_size)
        bad_col_sums = np.zeros(n_col)
        for x in range(n_col):  # jwst 指标先y后x
            bad_col_sums[x] = np.sum(photo_zero_mask[:,x*mosaic_size+edge_width:(x+1)*mosaic_size+edge_width])
        top_50_percent_count = int(n_col * col_ratio)
        # 获取排序后的索引
        sorted_indices = np.argsort(bad_col_sums)[::-1]  # 降序排序索引
        top_50_percent_indices = sorted_indices[:top_50_percent_count]  # 取前10%的索引
        for x in top_50_percent_indices:
            correlated_mask[:,x*mosaic_size+edge_width:(x+1)*mosaic_size+edge_width] = True

    # brightness_mask排除剩余的过亮点以及周围
    brightness_mask = np.zeros((2048, 2048), dtype=bool)

    if c_brightness:
        for y in range(2048):
            for x in range(2048):
                if  np.abs(pre_mask[y,x]) > brightness_thr: # not mask[y,x] and not dq_map_2d[y,x]
                    y_min = np.max([y-brightness_size,0])
                    x_min = np.max([x-brightness_size,0])
                    y_max = np.min([y+brightness_size,2048-1])
                    x_max = np.min([x+brightness_size,2048-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= brightness_size**2:
                                brightness_mask[y2,x2] = True

    mask = bad_mask + edge_mask + brightness_mask + correlated_mask

    delta_ratio = 1 - (total_pix - bad_pix) / (total_pix - flag_pix)

    pixel_after_pipeline = total_pix - bad_pix
    pipeline_bad = dq_map_2d + bad_mask
    pixel_after_pipeline_bad = total_pix - np.sum(pipeline_bad>0)
    bad_ratio = 1 - pixel_after_pipeline_bad/pixel_after_pipeline

    pipeline_bad_edge = dq_map_2d + bad_mask + edge_mask
    pixel_after_pipeline_bad_edge = total_pix - np.sum(pipeline_bad_edge>0)
    edge_ratio = 1 - pixel_after_pipeline_bad_edge/pixel_after_pipeline_bad

    pipeline_bad_edge_correlated = dq_map_2d + bad_mask + edge_mask + correlated_mask
    pixel_after_pipeline_bad_edge_correlated = total_pix - np.sum(pipeline_bad_edge_correlated>0)
    correlated_ratio = 1 - pixel_after_pipeline_bad_edge_correlated/pixel_after_pipeline_bad_edge

    pip_bad_edge_correlated_bright = dq_map_2d + bad_mask + edge_mask + correlated_mask + brightness_mask
    pixel_after_pip_bad_edge_correlated_bright = total_pix - np.sum(pip_bad_edge_correlated_bright>0)
    bright_ratio = 1 - pixel_after_pip_bad_edge_correlated_bright / pixel_after_pipeline_bad_edge_correlated

    data_behav_mask[i,0] = flag_pix / total_pix
    data_behav_mask[i,1] = delta_ratio
    data_behav_mask[i,2] = bad_ratio
    data_behav_mask[i,3] = edge_ratio
    data_behav_mask[i,4] = correlated_ratio
    data_behav_mask[i,5] = bright_ratio
    data_behav_mask[i,6] = pixel_after_pip_bad_edge_correlated_bright / 2048**2

    # 现在mask和原bad pixel mask合并
    # 加上 gain！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
    masked_indexes = np.where(mask)
    raw_photo_3 = (jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]) * gain
    raw_photo_3[dq_map_indexes] = np.nan
    raw_photo_3[masked_indexes] = np.nan
    raw_photo_3[low_gain_indexes] = np.nan
    raw_photo_flatten_3 = raw_photo_3.flatten()
    nan_mask_3 = np.isnan(raw_photo_flatten_3)
    photo_masked = raw_photo_flatten_3[~nan_mask_3]
    #masked_counts, masked_bin_edges = np.histogram(photo_masked, bins=range(dn_min, dn_max+1))
    raw_photo_2 = (jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]) * gain
    raw_photo_2[dq_map_indexes] = np.nan
    raw_photo_2[low_gain_indexes] = np.nan
    raw_photo_flatten_2 = raw_photo_2.flatten()
    nan_mask_2 = np.isnan(raw_photo_flatten_2)
    photo_good_flatten = raw_photo_flatten_2[~nan_mask_2]
    plt.figure(figsize=(12, 5))
    good_counts, good_bin_edges, good_patched = plt.hist(photo_good_flatten, bins=dn_hist_bins, alpha=0.8, color='blue',edgecolor='black',log=True)
    masked_counts, masked_bin_edges, masked_patched = plt.hist(photo_masked, bins=dn_hist_bins, alpha=0.8, color='red',edgecolor='black',log=True)
    # 设置边缘颜色的透明度
    for patch in plt.gca().patches:
        patch.set_edgecolor((1, 1, 1, 0.2))  # RGBA格式 (红, 绿, 蓝, 透明度)
    plt.title('Pixel Value Distribution')
    plt.xlabel('DN')
    plt.ylabel('Number of Pixels')
    plt.ylim(1,1e6)
    legend_elements = [
        Line2D([0], [0], color='b', lw=2, label='only JWST pipeline flags'),
        Line2D([0], [0], color='r', lw=2, label='w/ halo mask')
    ]
    plt.legend(handles=legend_elements, fontsize=12, loc='upper right', title_fontsize='13', frameon=True)
    good_peak   = np.argmax(good_counts) + dn_min
    masked_peak = np.argmax(masked_counts) + dn_min
    plt.savefig('./results/masked/' + input_file_base + '_histogram_after_mask.jpg')
    plt.close()

    # 保存结果
    result = np.array([range(dn_min,dn_max), masked_counts]).T
    np.savetxt('./results/masked/'+ input_file_base +'.txt',result, header = '{}'.format(bad_pix / total_pix))

    print(('{} pixels ({:.2f}% of the detector) has been flagged'.format(bad_pix, 100. * bad_pix / total_pix)))
    print('{}% pixels left after mask'.format(np.sum(masked_counts)/np.sum(good_counts)*100))
    print('peak of flagged but unmasked image at {}'.format(good_peak))
    print('peak of flagged and masked image at {}'.format(masked_peak))
    print(input_file_base + ' analysis completed')
    print(i)
    i += 1

group number is 245
1275427 pixels (30.41% of the detector) has been flagged
9.008779010294846% pixels left after mask
peak of flagged but unmasked image at 15
peak of flagged and masked image at 6
jw01121144001_02102_00001_nrs2 analysis completed
0
group number is 245
1288306 pixels (30.72% of the detector) has been flagged
8.19997658533322% pixels left after mask
peak of flagged but unmasked image at 23
peak of flagged and masked image at 23
jw01121156001_02102_00001_nrs2 analysis completed
1
group number is 245
1344978 pixels (32.07% of the detector) has been flagged
6.573578792059001% pixels left after mask
peak of flagged but unmasked image at 19
peak of flagged and masked image at 14
jw01121154001_02102_00001_nrs2 analysis completed
2
group number is 245
1289054 pixels (30.73% of the detector) has been flagged
8.515284603733656% pixels left after mask
peak of flagged but unmasked image at 13
peak of flagged and masked image at 9
jw01121004001_02102_00001_nrs2 analysis completed
3

In [ ]:
np.savetxt('./results/data_behav.txt',data_behav_mask)

In [ ]:
#data_behav_mask[i,0] = flag_pix / total_pix
#data_behav_mask[i,1] = delta_ratio
#data_behav_mask[i,2] = bad_ratio
#data_behav_mask[i,3] = edge_ratio
#data_behav_mask[i,4] = correlated_ratio
#data_behav_mask[i,5] = bright_ratio
#data_behav_mask[i,6] = pixel_after_pip_bad_edge_correlated_bright / 2048**2

In [ ]:
print(np.average(data_behav_mask[:,5]))
print(np.std(data_behav_mask[:,5], ddof=1))

In [ ]:
testdata = (1-data_behav_mask[:,0]) * (1-data_behav_mask[:,1]) * (1-data_behav_mask[:,2]) * (1-data_behav_mask[:,3])  * (1-data_behav_mask[:,4]) * (1-data_behav_mask[:,5])
#testdata = data_behav_mask[:,5]
print(np.average(testdata))
print(np.std(testdata, ddof=1))

In [ ]:
# unmaksed, 过完pipeline屏蔽掉flagged pixels之后直出
data_path = Path('../data/JWST/3000s_exposure')
for item in data_path.iterdir():
    input_file_base = item.name
    jump_file = '../data/JWST/3000s_exposure/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]

    raw_photo_2 = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    raw_photo_2[dq_map_indexes] = np.nan
    raw_photo_flatten_2 = raw_photo_2.flatten()
    nan_mask_2 = np.isnan(raw_photo_flatten_2)
    photo_good_flatten = raw_photo_flatten_2[~nan_mask_2]
    good_counts, good_bin_edges = np.histogram(photo_good_flatten, bins=range(dn_min, dn_max+1))

    # 保存结果
    result = np.array([range(dn_min,dn_max), good_counts]).T
    np.savetxt('./results/unmasked/'+ input_file_base +'_unmasked.txt',result)

    print(input_file_base + ' analysis completed')